# Deep Neural Decision Tree (DNDT) and Forest (DNDF) 3-Track Benchmark Suite

> **Benchmarking & Replication against:**  
> Rofiqul Islam, Nihad Karim Chowdhury, and Muhammad Ashad Kabir. *"Robust COVID-19 detection from cough sounds using deep neural decision tree and forest: A comprehensive cross-datasets evaluation."* Expert Systems with Applications, Vol. 310, 2026, 131235. [DOI: 10.1016/j.eswa.2026.131235](https://doi.org/10.1016/j.eswa.2026.131235)

---

## 🔬 The 3 Independent Research Tracks:
1. **Track 1: Authors' Exact Paper Reproduction** (10-Fold Stratified CV, Cough audio, RFECV 33 features, 25 Trees, Depth 11, LR 0.01, 14 Epochs)
2. **Track 2: Methodologically Corrected Reproduction** (Strict Nested 10-Fold CV with 0% data leakage: fold-only RFECV, fresh initialization, inner-validation threshold tuning)
3. **Track 3: COVID-RARS Clinical Reliability Protocol** (Participant-disjoint holdouts, true temporal validation, tripartite multimodal fusion with breath and speech, zero-shot COUGHVID external transfer)

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install core dependencies
%pip install -q torch scikit-learn imbalanced-learn pandas numpy matplotlib seaborn optuna librosa

## 2. Sync Repository & Fresh Module Reload

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Clone / Reset repository to origin/main
repo_path = Path("/content/Covid-RARS") if Path("/content").exists() else Path("/kaggle/working/Covid-RARS")
if not repo_path.exists():
    !git clone https://github.com/ishaaaan17/Covid-RARS.git {repo_path}
else:
    subprocess.run(f"git -C {repo_path} fetch origin && git -C {repo_path} reset --hard origin/main", shell=True)

# Set active working directory to repo root
os.chdir(str(repo_path))

# Add src to Python path & clear module cache
sys.path.insert(0, str(repo_path / "src"))
for k in list(sys.modules.keys()):
    if 'covid_rars' in k:
        del sys.modules[k]

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Repository synced cleanly. Working directory: {os.getcwd()} | Device: {device}")


## 3. Load Engineered Acoustic Feature Banks

In [ ]:
import pandas as pd

# Mount Drive if on Colab
if Path("/content").exists() and not Path("/content/drive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# Discover dataset
candidate_paths = [
    Path("/content/drive/MyDrive/BTP/processed/features_compare_is10_merged.csv"),
    Path("/kaggle/input/covid-rars-features/features_compare_is10_merged.csv"),
    Path("data/processed/features_compare_is10_merged.csv"),
    Path("D:/BTP/processed/features_compare_is10_merged.csv"),
]

feat_file = next((p for p in candidate_paths if p.exists()), None)
if feat_file is None:
    raise FileNotFoundError("Please place features_compare_is10_merged.csv in Google Drive or data/processed/")

print(f"Loading features from {feat_file}...")
features_df = pd.read_csv(feat_file, low_memory=False)
print(f"Loaded {len(features_df)} recordings across modalities: {list(features_df['modality'].unique())}")

## 3.5 ⚡ Fast 15-Second Smoke Test (Preflight Validation)

Run this cell to verify all 3 tracks (Track 1, Track 2, and Track 3) end-to-end on a small 50-sample subset before executing full-scale runs.

In [ ]:
# Run preflight smoke test across all 3 tracks in <15 seconds
!python scripts/79_run_dndf_reliability.py --smoke-test --device {device}


## 4. Track 1: Authors' Exact Paper Reproduction Benchmark

Matches Islam et al. (*Expert Systems with Applications*, 2026 / arXiv:2501.01117):
- **Modality:** Cough Audio
- **Cross-Validation:** 10-Fold Stratified Recording-Level CV
- **Feature Selection:** ExtraTrees + RFECV selecting ~33 features on Coswara
- **Model:** DNDF (25 trees, depth 11, lr 0.01, batch size 16, 14 epochs, SMOTE balancing)

In [ ]:
from covid_rars.dndf_tracks import run_track1_author_exact_reproduction

track1_summary = run_track1_author_exact_reproduction(
    features_df=features_df,
    modality="cough",
    n_splits=10,
    num_trees=25,
    depth=11,
    learning_rate=0.01,
    batch_size=16,
    max_epochs=14,
    n_selected_features=33,
    device=device,
)

## 5. Track 2: Methodologically Corrected Leak-Free Reproduction

Strict scientific audit of the author's architecture:
- Feature selection fitted **strictly** on the training fold.
- Threshold tuned on **inner-validation** split (0% test exposure).
- Fresh model & scaler initialization per fold.

In [ ]:
from covid_rars.dndf_tracks import run_track2_corrected_leak_free_reproduction

track2_summary = run_track2_corrected_leak_free_reproduction(
    features_df=features_df,
    modality="cough",
    n_splits=10,
    num_trees=25,
    depth=11,
    learning_rate=0.01,
    batch_size=16,
    max_epochs=14,
    n_selected_features=33,
    device=device,
)

## 6. Track 3: COVID-RARS Clinical Reliability Suite & Multimodal Fusion

Evaluates the differentiable decision forest on:
- **Track 3A:** 10 Repeated Participant-Disjoint Holdouts
- **Track 3B:** Chronological Temporal Validation (real calendar dates only)
- **Track 3C:** Zero-Shot External Generalization (COUGHVID)
- **Track 3D:** Complete-Case Multimodal Fusion (Cough + Breath + Speech via Stacked Logistic)

In [ ]:
from covid_rars.dndf_tracks import run_track3_covid_rars_reliability_suite

track3_results = run_track3_covid_rars_reliability_suite(
    features_df=features_df,
    external_features_df=None,
    modalities=["cough", "breath", "speech"],
    seeds=[1, 2, 5, 12, 40],
    num_trees=25,
    depth=5,
    learning_rate=0.005,
    max_epochs=30,
    device=device,
)